In [1]:
import rasterio
import numpy as np
import geopandas as gpd
import os
import imageio

from PIL import Image
from tqdm import tqdm
from shapely.geometry import LineString
from skimage.morphology import skeletonize
from skimage.measure import find_contours

In [2]:
# -------------------------
# 📂 PATHS
# -------------------------
raster_path = "data/el_harrach_georef.tif"
legend_dir = "data/legend_line_clean"
output_dir = "output/vect/lines"
os.makedirs(output_dir, exist_ok=True)

# -------------------------
# 🎯 LEGEND CLEANING SETTINGS
# -------------------------
BG_COLOR = np.array([241, 238, 232])  # #f1eee8
BG_TOLERANCE = 0

In [3]:
# -------------------------
# 🎨 DOMINANT COLOR (CLEAN LEGEND)
# -------------------------
def get_dominant_rgb_clean(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((100, 100))

    arr = np.array(img).reshape(-1, 3)

    # remove background pixels
    mask = np.linalg.norm(arr - BG_COLOR, axis=1) > BG_TOLERANCE
    filtered = arr[mask]

    if len(filtered) == 0:
        raise ValueError(f"No valid pixels in {image_path}")

    return np.median(filtered, axis=0).astype(np.uint8)

In [4]:

# -------------------------
# 🧠 BUILD COLOR → CLASS MAP
# -------------------------
color_class_map = {}

print("🎨 Extracting line legend colors...\n")

for file in os.listdir(legend_dir):
    if not file.lower().endswith(".png"):
        continue

    path = os.path.join(legend_dir, file)
    class_name = os.path.splitext(file)[0]

    try:
        rgb = get_dominant_rgb_clean(path)
        color_class_map[tuple(rgb)] = class_name
        print(f"{class_name}: {rgb}")

    except Exception as e:
        print(f"⚠️ Skipping {file}: {e}")

print("\n✅ Color map ready\n")

🎨 Extracting line legend colors...

Minor_power_line___Path_from_tee_area_to_the_green_of_a_golf_course: [242 239 233]
Living_street: [237 234 226]
Access_road__may_be_also_outside_of_a_city: [237 234 226]
Living_street_under_construction: [241 238 233]
The_link_roads__sliproads___ramps__leading_to_and_from_a_trunk_highway: [237 234 226]
Sub-national_boundary__fourth-highest_level: [242 239 233]
Miniature_railway: [242 239 233]
River___Canal: [242 239 233]
Residential_road_only_local_traffic: [242 239 233]
Cycleway: [241 238 233]
Taxiway: [242 239 233]
Subordinated_way_in_a_parking_lot___drive-through_highway___driveway___slipway: [237 234 226]
Track__Solid_surface: [242 239 233]
River_intermittent___Canal_intermittent___River_seasonal___Canal_seasonal: [242 239 233]
Embankment: [242 239 233]
Trunks__the_most_important_roads_in_a_road_network_that_aren_t_motorways: [237 234 226]
Stream_in_pipe_or_tunnel___Ditch_in_pipe_or_tunnel___drain_in_pipe_or_tunnel: [242 239 233]
Motorway__the_mo

In [5]:
# -------------------------
# 📥 READ RASTER
# -------------------------
with rasterio.open(raster_path) as src:
    img = src.read()
    transform = src.transform
    crs = src.crs

img = np.transpose(img, (1, 2, 0))[:, :, :3].astype(np.uint8)

In [ ]:

# -------------------------
# 🚀 PROCESS EACH CLASS
# -------------------------
for rgb, class_name in tqdm(color_class_map.items(), desc="Processing line classes"):

    target = np.array(rgb, dtype=np.int16)

    # color match mask
    mask = np.all(np.abs(img.astype(np.int16) - target) <= 0, axis=2)

    if not np.any(mask):
        print(f"⚠️ Skipping {class_name}: empty mask")
        continue

    # -------------------------
    # 🧵 LINE EXTRACTION (SKELETON + CONTOURS)
    # -------------------------
    skel = skeletonize(mask > 0)
    contours = find_contours(skel.astype(np.uint8), 0.5)

    geoms = []

    for contour in contours:
        if len(contour) < 2:
            continue

        coords = []
        for y, x in contour:
            lon, lat = rasterio.transform.xy(transform, y, x)
            coords.append((lon, lat))

        if len(coords) >= 2:
            geoms.append(LineString(coords))

    if not geoms:
        print(f"⚠️ Skipping {class_name}: no valid line geometry")
        continue

    # -------------------------
    # 💾 SAVE GEOJSON
    # -------------------------
    gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
    gdf["class"] = class_name

    geojson_path = os.path.join(output_dir, f"{class_name}.geojson")
    gdf.to_file(geojson_path, driver="GeoJSON")

    # -------------------------
    # 🖼️ DEBUG OVERLAY
    # -------------------------
    overlay = img.copy()

    highlight = np.zeros_like(img)
    highlight[:, :, 0] = 255  # red highlight

    alpha = 0.5
    overlay[mask] = (
        (1 - alpha) * overlay[mask] + alpha * highlight[mask]
    ).astype(np.uint8)

    overlay_path = os.path.join(output_dir, f"{class_name}_overlay.png")
    imageio.imwrite(overlay_path, overlay)

print("\n✅ DONE: Line extraction complete")

Processing line classes:   0%|          | 0/12 [00:00<?, ?it/s]